# Notebook 6 — Visualizations and Final Results

Generate all final visualizations for the project report.

## Visualizations Generated
1. Spatial flood risk map for Dhemaji
2. Model version progression chart
3. Feature importance bar chart
4. Combined ROC curve
5. Flood probability distribution

## 1. Imports and Load

In [ ]:
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 100

# Load data and model
df = pd.read_csv("../data/processed/dhemaji_flood_FINAL.csv")
df["date"] = pd.to_datetime(df["date"])
df["year"] = df["date"].dt.year

model = joblib.load("../models/flood_model_FINAL.pkl")

features = [
    "dist_to_major_river",
    "elevation",
    "tree_cover",
    "slope",
    "rain_anomaly",
    "rain_5day",
    "rain_3day",
    "rainfall_mm",
    "runoff_sum",
    "runoff_anomaly"
]

## 2. Spatial Flood Risk Map

Compute average flood probability per grid cell and visualize on a spatial map.

In [ ]:
# Compute flood probability for each row
df["flood_proba"] = model.predict_proba(df[features])[:,1]

# Aggregate to per-cell flood rate
spatial = df.groupby(
    ["latitude","longitude"]
).agg(
    flood_rate = ("flood_label", "mean"),
    avg_proba  = ("flood_proba", "mean")
).reset_index()

print("Unique cells:", len(spatial))
print("Flood rate stats:")
print(spatial["flood_rate"].describe().round(3))

In [ ]:
fig, ax = plt.subplots(figsize=(12,8))
scatter = ax.scatter(
    spatial["longitude"],
    spatial["latitude"],
    c=spatial["flood_rate"],
    cmap="RdYlGn_r",
    s=5, alpha=0.7
)
plt.colorbar(scatter, ax=ax, label="Average Flood Rate")
ax.set_title("Dhemaji District — Flood Risk Map (2019-2024)")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
plt.tight_layout()
plt.savefig("../figures/flood_risk_map.png", dpi=150)
plt.show()

## 3. Predicted Flood Probability Map (Model Output)

In [ ]:
fig, ax = plt.subplots(figsize=(12,8))
scatter = ax.scatter(
    spatial["longitude"],
    spatial["latitude"],
    c=spatial["avg_proba"],
    cmap="RdYlGn_r",
    s=5, alpha=0.7
)
plt.colorbar(scatter, ax=ax, label="Model Predicted Probability")
ax.set_title("Model Predicted Flood Probability — Dhemaji")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
plt.tight_layout()
plt.savefig("../figures/predicted_flood_probability.png", dpi=150)
plt.show()

## 4. Model Version Progression

In [ ]:
versions = pd.DataFrame([
    {"Version":"V1\nOriginal",     "Precision":0.56, "Recall":0.84, "F1":0.67, "AUC":0.950},
    {"Version":"V2\nSAR+Distance", "Precision":0.71, "Recall":0.92, "F1":0.80, "AUC":0.976},
    {"Version":"V3\nUpstream",     "Precision":0.70, "Recall":0.92, "F1":0.80, "AUC":0.976},
    {"Version":"V4\nMultiyear",    "Precision":0.77, "Recall":0.91, "F1":0.84, "AUC":0.992},
    {"Version":"V5\nReduced",      "Precision":0.74, "Recall":0.94, "F1":0.82, "AUC":0.992},
    {"Version":"FINAL\nGB Tuned",  "Precision":0.85, "Recall":0.82, "F1":0.83, "AUC":0.990}
])

x = np.arange(len(versions))
width = 0.2

fig, ax = plt.subplots(figsize=(14,6))
ax.bar(x - 1.5*width, versions["Precision"], width, label="Precision", color="steelblue")
ax.bar(x - 0.5*width, versions["Recall"],    width, label="Recall",    color="coral")
ax.bar(x + 0.5*width, versions["F1"],        width, label="F1",        color="green")
ax.bar(x + 1.5*width, versions["AUC"],       width, label="ROC-AUC",   color="purple")

ax.set_xticks(x)
ax.set_xticklabels(versions["Version"])
ax.set_ylabel("Score")
ax.set_title("Model Performance Across Versions")
ax.legend(loc="lower right")
ax.set_ylim(0, 1.1)
plt.tight_layout()
plt.savefig("../figures/version_progression.png", dpi=150)
plt.show()

## 5. Feature Importance Bar Chart

In [ ]:
importance = pd.DataFrame({
    "Feature":    features,
    "Importance": model.feature_importances_
}).sort_values("Importance", ascending=True)

fig, ax = plt.subplots(figsize=(11,7))
ax.barh(
    importance["Feature"],
    importance["Importance"],
    color="steelblue",
    edgecolor="black"
)
ax.set_title("Feature Importance — Final Model")
ax.set_xlabel("Importance Score")

for i, v in enumerate(importance["Importance"]):
    ax.text(v + 0.005, i, f"{v:.3f}", va="center", fontsize=9)

plt.tight_layout()
plt.savefig("../figures/feature_importance_final.png", dpi=150)
plt.show()

## 6. Flood Probability Distribution

In [ ]:
test = df[df["year"] == 2024]
test_proba = model.predict_proba(test[features])[:,1]

fig, ax = plt.subplots(figsize=(12,5))
ax.hist(
    test_proba[test["flood_label"] == 0],
    bins=50, alpha=0.6, color="steelblue",
    label="Actual No Flood"
)
ax.hist(
    test_proba[test["flood_label"] == 1],
    bins=50, alpha=0.6, color="coral",
    label="Actual Flood"
)
ax.axvline(0.4, color="black", linestyle="--",
           label="Decision Threshold (0.4)")
ax.set_title("Predicted Probability Distribution by Actual Class")
ax.set_xlabel("Predicted Flood Probability")
ax.set_ylabel("Frequency (log scale)")
ax.set_yscale("log")
ax.legend()
plt.tight_layout()
plt.savefig("../figures/probability_distribution.png", dpi=150)
plt.show()

## 7. Summary Dashboard

Single figure summarizing all key results.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16,11))

# Panel 1 — Feature importance
imp_sorted = importance.sort_values("Importance", ascending=True)
axes[0,0].barh(imp_sorted["Feature"], imp_sorted["Importance"], color="steelblue")
axes[0,0].set_title("Feature Importance")
axes[0,0].set_xlabel("Importance")

# Panel 2 — Flood by year
flood_year = df.groupby("year")["flood_label"].mean()
axes[0,1].bar(flood_year.index, flood_year.values, color="coral", edgecolor="black")
axes[0,1].set_title("Flood Rate Per Year")
axes[0,1].set_xlabel("Year")
axes[0,1].set_ylabel("Flood Rate")

# Panel 3 — Spatial map
scatter = axes[1,0].scatter(
    spatial["longitude"], spatial["latitude"],
    c=spatial["flood_rate"], cmap="RdYlGn_r", s=3
)
axes[1,0].set_title("Spatial Flood Risk")
axes[1,0].set_xlabel("Longitude")
axes[1,0].set_ylabel("Latitude")
plt.colorbar(scatter, ax=axes[1,0], label="Flood Rate")

# Panel 4 — Version progression F1
axes[1,1].plot(
    range(len(versions)), versions["F1"],
    marker="o", linewidth=2, color="steelblue", markersize=10
)
axes[1,1].set_xticks(range(len(versions)))
axes[1,1].set_xticklabels(versions["Version"], rotation=0, fontsize=8)
axes[1,1].set_title("F1 Score Progression Across Versions")
axes[1,1].set_ylabel("F1 Score")
axes[1,1].set_ylim(0.6, 0.9)
axes[1,1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("../figures/summary_dashboard.png", dpi=150)
plt.show()

## Files Generated

All figures saved to `../figures/`:
- flood_risk_map.png — Historical flood rate map
- predicted_flood_probability.png — Model output map
- version_progression.png — Improvement over versions
- feature_importance_final.png — Final feature ranking
- probability_distribution.png — Class separation
- summary_dashboard.png — Combined summary figure